In [1]:
import csv
import difflib
import gc
import math
import os
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

# Machine Learning & Tensors
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

# Semantic Web / RDFLib
from rdflib import Graph, RDF, RDFS, URIRef, term, Namespace
from rdflib.namespace import OWL, SKOS

# Prevent rdflib from raising an exception on invalid lexical forms
term._fail_on_invalid_lexical_form = False

# Natural Language Toolkit (NLTK)
try:
    import nltk
    from nltk.corpus import wordnet as wn
except ImportError:
    pass

c:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0702 02:28:36.676000 17768 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0702 02:28:36.723000 17768 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:

# Define common namespaces
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")

class MOSAIC:

    def __init__(self, lex_thres=0.62, sem_thres=0.62, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.lex_thres = lex_thres
        self.sem_thres = sem_thres
        self.model = None  
        self._wn_ready = False
        self.model_name = model_name
        
        # --- CPU Thread Scaling Control ---
        cores = os.cpu_count() or 1
        if hasattr(os, 'sched_getaffinity'):
            try:
                cores = len(os.sched_getaffinity(0))
            except Exception:
                pass
        threads = max(4, cores - 1)
        torch.set_num_threads(threads)

    def init_wordnet(self):
        if not self._wn_ready:
            try:
                from nltk.corpus import wordnet as wn
                wn.synsets("test")
            except (ImportError, LookupError):
                import nltk
                print(" [MOSAIC] Downloading required NLTK resources...")
                nltk.download("wordnet", quiet=True)
                nltk.download("omw-1.4", quiet=True)
            self._wn_ready = True

    def init_model(self):
        if self.model is None:
            print(f" [MOSAIC] Initializing embedding engine on CPU: {self.model_name}")
            raw_model = SentenceTransformer(self.model_name, device="cpu")
            
            try:
                import torchao
                from torchao.quantization import quantize_
                from torchao.quantization.quant_api import int8_dynamic_activation_int8_weight
                
                print(" [MOSAIC] Applying modern torchao INT8 dynamic quantization.")
                quantize_(raw_model, int8_dynamic_activation_int8_weight())
                self.model = raw_model
            except (ImportError, Exception):
                try:
                    from torch.ao.quantization import quantize_dynamic
                    self.model = quantize_dynamic(raw_model, {torch.nn.Linear}, dtype=torch.qint8)
                except Exception:
                    self.model = raw_model

    def normalise_label(self, text: str) -> str:
        if not text: 
            return ""
        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', str(text))
        text = text.replace('_', ' ').replace('-', ' ').lower().strip()
        return " ".join(text.split())

    def get_text_data(self, uri, graph) -> str:
        # Check target languages explicitly present in OAEI cultural/thesaurus tracks
        target_langs = ['de', 'fr', 'sl', 'hr', 'en']
        
        for lang in target_langs:
            for label in graph.objects(uri, SKOS.prefLabel):
                if hasattr(label, 'language') and label.language == lang:
                    return self.normalise_label(label)
                    
            for label in graph.objects(uri, RDFS.label):
                if hasattr(label, 'language') and label.language == lang:
                    return self.normalise_label(label)

        # Fallback to any string value found if preferred language tags aren't explicitly matched
        label = graph.value(uri, RDFS.label) or graph.value(uri, SKOS.prefLabel)
        if label:
            return self.normalise_label(label)
            
        frag = str(uri).split('/')[-1].split('#')[-1]
        frag = re.sub(r'%[0-9A-Fa-f]{2}', ' ', frag)
        return self.normalise_label(frag)

    def load_ontology(self, path: Path) -> Graph:
        g = Graph()
        if path.suffix == ".ttl":
            formats = ["turtle", "xml"]
        elif path.suffix in [".owl", ".rdf", ".xml"]:
            formats = ["xml", "turtle"]
        else:
            formats = [None]

        for fmt in formats:
            try:
                g.parse(str(path), format=fmt)
                return g
            except Exception:
                continue
        
        try:
            g.parse(str(path))
            return g
        except Exception as e:
            print(f" [MOSAIC] Failed to load {path.name}: {e}")
            return None

    def extract_entities(self, graph: Graph):
        self.init_wordnet()
        from nltk.corpus import wordnet as wn

        classes = set(graph.subjects(RDF.type, OWL.Class)).union(set(graph.subjects(RDF.type, SKOS.Concept)))
        props = set(graph.subjects(RDF.type, OWL.ObjectProperty)).union(set(graph.subjects(RDF.type, OWL.DatatypeProperty)))
        all_subs = set(graph.subjects())
        insts = all_subs - classes - props
        
        insts = {i for i in insts if isinstance(i, URIRef) and "oboInOwl" not in str(i)}
        classes = {c for c in classes if isinstance(c, URIRef) and "oboInOwl" not in str(c)}
        props = {p for p in props if isinstance(p, URIRef) and "oboInOwl" not in str(p)}

        entities = {}
        
        # OMW language code extensions map
        wn_langs = ['eng', 'deu', 'fra', 'slv', 'hrv']
        
        for entity_set, etype in [(classes, OWL.Class), (props, OWL.ObjectProperty), (insts, OWL.NamedIndividual)]:
            for s in entity_set:
                lbl_str = self.get_text_data(s, graph)
                tokens = set(lbl_str.split())
                
                syns = set()
                for t in tokens:
                    if len(t) > 2:
                        for lang_code in wn_langs:
                            try:
                                for syn in wn.synsets(t, lang=lang_code):
                                    for lm in syn.lemmas(lang=lang_code):
                                        syns.add(lm.name().lower().replace("_", " "))
                            except Exception:
                                continue

                entities[s] = {
                    "label": lbl_str,
                    "type": etype,
                    "tokens": tokens,
                    "synonyms": syns
                }
        return entities

    def build_inverted_index(self, labels):
        index = defaultdict(list)
        counts = defaultdict(int)
        for label in labels:
            for word in set(label.split()):
                if len(word) > 2:
                    counts[word] += 1
                    
        max_limit = max(50, int(len(labels) * 0.05))
        for idx, label in enumerate(labels):
            for word in label.split():
                if len(word) > 2 and counts[word] <= max_limit:
                    index[word].append(idx)
        return index

    def fast_token_match(self, src_label, tgt_labels, inverted_index):
        src_words = src_label.split()
        if not src_words: return 0.0, 0
        
        counts = defaultdict(int)
        for word in src_words:
            if word in inverted_index:
                for idx in inverted_index[word]:
                    counts[idx] += 1
                    
        if not counts:
            return 0.0, 0
            
        best_idx = max(counts, key=counts.get)
        match_count = counts[best_idx]
        max_words = max(len(src_words), len(tgt_labels[best_idx].split()))
        return (match_count / max_words if max_words > 0 else 0.0), best_idx

    def semantic_similarity(self, filtered_src, filtered_tgt, batch_size=512):
        if not filtered_src or not filtered_tgt:
            return []

        self.init_model()
        
        src_uris = list(filtered_src.keys())
        src_labels = [meta["label"] for meta in filtered_src.values()]
        
        tgt_uris = list(filtered_tgt.keys())
        tgt_labels = [meta["label"] for meta in filtered_tgt.values()]
        
        tgt_index = self.build_inverted_index(tgt_labels)
        
        emb2 = self.model.encode(tgt_labels, convert_to_tensor=True, show_progress_bar=False, batch_size=batch_size)
        emb1_all = self.model.encode(src_labels, convert_to_tensor=True, show_progress_bar=False, batch_size=batch_size)

        candidates = []
        
        with torch.inference_mode():
            for local_i, (s_uri, s_meta) in enumerate(zip(src_uris, src_labels)):
                s_lbl = filtered_src[s_uri]["label"]
                s_type = filtered_src[s_uri]["type"]
                s_words = s_lbl.split()
                
                if len(tgt_labels) > 300:
                    cand_indices = set()
                    for word in s_words:
                        if word in tgt_index:
                            cand_indices.update(tgt_index[word])
                    if not cand_indices:
                        continue
                    block_list = sorted(list(cand_indices))
                else:
                    block_list = list(range(len(tgt_labels)))

                emb2_block = emb2[block_list]
                emb1_item = emb1_all[local_i].unsqueeze(0)
                
                sim_scores = util.cos_sim(emb1_item, emb2_block)[0]
                best_local_idx = torch.argmax(sim_scores).item()
                score = sim_scores[best_local_idx].item()
                best_tgt_idx = block_list[best_local_idx]
                
                t_uri = tgt_uris[best_tgt_idx]
                t_lbl = tgt_labels[best_tgt_idx]
                t_type = filtered_tgt[t_uri]["type"]

                if s_type != t_type:
                    continue

                ratio = difflib.SequenceMatcher(None, s_lbl, t_lbl).quick_ratio()
                if len(tgt_labels) > 300 and ratio < 0.45 and score < (self.sem_thres + 0.1):
                    score *= 0.75

                if score < self.sem_thres:
                    t_score, fast_idx = self.fast_token_match(s_lbl, tgt_labels, tgt_index)
                    if t_score > 0.65 and t_score > score:
                        score = t_score
                        best_tgt_idx = fast_idx
                        t_uri = tgt_uris[best_tgt_idx]
                        t_lbl = tgt_labels[best_tgt_idx]

                if score >= self.sem_thres:
                    candidates.append({
                        "source": s_uri,
                        "target": t_uri,
                        "type": s_type,
                        "combined_score": score
                    })
                    
        return candidates

    def align(self, src_graph: Graph, tgt_graph: Graph):
        src_ents = self.extract_entities(src_graph)
        tgt_ents = self.extract_entities(tgt_graph)
        
        final_pool = []
        claimed_src = set()
        claimed_tgt = set()
        
        tgt_lookup = {meta["label"]: uri for uri, meta in tgt_ents.items() if meta["label"]}
            
        for s_uri, s_meta in src_ents.items():
            s_lbl = s_meta["label"]
            if s_lbl in tgt_lookup:
                t_uri = tgt_lookup[s_lbl]
                t_meta = tgt_ents[t_uri]
                if s_meta["type"] == t_meta["type"]:
                    claimed_src.add(s_uri)
                    claimed_tgt.add(t_uri)
                    final_pool.append({
                        "source": s_uri,
                        "target": t_uri,
                        "type": s_meta["type"],
                        "combined_score": 1.0
                    })

        filtered_src = {k: v for k, v in src_ents.items() if k not in claimed_src}
        filtered_tgt = {k: v for k, v in tgt_ents.items() if k not in claimed_tgt}

        if filtered_src and filtered_tgt:
            sem_candidates = self.semantic_similarity(filtered_src, filtered_tgt)
            
            sorted_pairs = sorted(sem_candidates, key=lambda x: x["combined_score"], reverse=True)
            for c in sorted_pairs:
                if c["source"] not in claimed_src and c["target"] not in claimed_tgt:
                    claimed_src.add(c["source"])
                    claimed_tgt.add(c["target"])
                    final_pool.append(c)

        alignments = set()
        eq_class = "http://www.w3.org/2002/07/owl#equivalentClass"
        eq_prop = "http://www.w3.org/2002/07/owl#equivalentProperty"
        same_as = "http://www.w3.org/2002/07/owl#sameAs"

        for c in final_pool:
            s_uri, t_uri, etype = c["source"], c["target"], c["type"]
            if etype == OWL.Class:
                alignments.add((str(s_uri), eq_class, str(t_uri)))
            elif etype in [OWL.ObjectProperty, OWL.DatatypeProperty]:
                alignments.add((str(s_uri), eq_prop, str(t_uri)))
            else:
                alignments.add((str(s_uri), same_as, str(t_uri)))

        return alignments


class OAEITrackRunner:

    def __init__(self, matcher: MOSAIC):
        self.matcher = matcher
        self.log = []

    def load_reference_alignments(self, path: Path) -> set:
        ref_set = set()
        g = Graph()
        try:
            g.parse(str(path), format="turtle")
            valid_preds = {
                "http://www.w3.org/2002/07/owl#equivalentClass",
                "http://www.w3.org/2000/01/rdf-schema#subClassOf",
                "http://www.w3.org/2002/07/owl#equivalentProperty",
                "http://www.w3.org/2000/01/rdf-schema#subPropertyOf",
                "http://www.w3.org/2002/07/owl#sameAs",
            }
            for s, p, o in g:
                if str(p) in valid_preds:
                    nodes = sorted([str(s), str(o)])
                    ref_set.add((nodes[0], str(p), nodes[1]))
        except Exception as e:
            print(f" Could not read reference file {path.name}: {e}")
        return ref_set

    def serialize_alignments_to_ttl(self, alignments: set, path: Path):
        g = Graph()
        for src, pred, tgt in alignments:
            g.add((URIRef(src), URIRef(pred), URIRef(tgt)))
        try:
            g.serialize(destination=str(path), format="turtle")
            print(f"   [MOSAIC] Output saved to: {path.parent.name}/{path.name}")
        except Exception as e:
            print(f"   [MOSAIC] Serialization error: {e}")

    def calculate_metrics(self, sys_align, ref_align):
        if not ref_align:
            return 0.0, 0.0, 0.0

        sys_canon = set()
        for s, p, o in sys_align:
            nodes = sorted([str(s), str(o)])
            sys_canon.add((nodes[0], str(p), nodes[1]))

        tp = len(sys_canon.intersection(ref_align))
        p = tp / len(sys_canon) if sys_canon else 0.0
        r = tp / len(ref_align) if ref_align else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

        return round(p, 4), round(r, 4), round(f1, 4)

    def find_ontology_file(self, folder: Path, name: str) -> Path:
        for ext in [".owl", ".rdf", ".ttl", ".xml"]:
            p = folder / f"{name}{ext}"
            if p.exists():
                return p
        return None

    def run_all_tracks(self, base_dir: str, csv_out: str = "mosaic_evaluation_report.csv"):
        start_global = time.time()
        base_path = Path(base_dir)
        res_dir = Path("../results")
        res_dir.mkdir(parents=True, exist_ok=True)

        if not base_path.exists():
            print(f"Error: Base directory '{base_dir}' does not exist.")
            return

        for track in base_path.iterdir():
            if not track.is_dir():
                continue

            print(f"\n" + "=" * 50)
            print(f" TRACK RUNNER: {track.name.upper()}")
            print(f"=" * 50)

            tasks = list(track.glob("*.ttl"))
            p_sum, r_sum, f_sum, t_sum = 0.0, 0.0, 0.0, 0.0
            count = 0

            for tf in tasks:
                parts = tf.stem.split("-")
                if len(parts) != 2:
                    if "human-mouse" in tf.stem:
                        parts = ["human", "mouse"]
                    else:
                        continue

                ont_folder = track / "ontologies"
                src_p = self.find_ontology_file(ont_folder, parts[0])
                tgt_p = self.find_ontology_file(ont_folder, parts[1])

                print(f"\nMOSAIC Task: {parts[0]} ➔ {parts[1]}")

                if not src_p or not tgt_p:
                    print(" Skipping task. Missing ontology file.")
                    continue

                ref_align = self.load_reference_alignments(tf)
                src_g = self.matcher.load_ontology(src_p)
                tgt_g = self.matcher.load_ontology(tgt_p)

                if src_g and tgt_g:
                    t0 = time.time()
                    alignments = self.matcher.align(src_g, tgt_g)
                    dt = round(time.time() - t0, 2)
                    
                    print(f" Step complete. MOSAIC returned {len(alignments)} matches in {dt}s.")

                    out_ttl = res_dir / f"mosaic_{track.name}_{tf.name}"
                    self.serialize_alignments_to_ttl(alignments, out_ttl)

                    p, r, f1 = self.calculate_metrics(alignments, ref_align)
                    print(f"   Metrics -> Precision: {p}, Recall: {r}, F1-Score: {f1} (Time: {dt}s)")

                    self.log.append({
                        "Track": track.name,
                        "Task": tf.stem,
                        "Precision": p,
                        "Recall": r,
                        "F1-Score": f1,
                        "Time (s)": dt,
                        "Type": "Task",
                    })

                    p_sum += p
                    r_sum += r
                    f_sum += f1
                    t_sum += dt
                    count += 1

                    del src_g, tgt_g
                    gc.collect()

            if count > 0:
                avg_p = round(p_sum / count, 4)
                avg_r = round(r_sum / count, 4)
                avg_f1 = round(f_sum / count, 4)
                avg_t = round(t_sum / count, 2)

                print(f"\n Track [{track.name}] AVERAGES -> P: {avg_p}, R: {avg_r}, F1: {avg_f1} | Avg Time: {avg_t}s")

                self.log.append({
                    "Track": track.name,
                    "Task": "TRACK_AVERAGE",
                    "Precision": avg_p,
                    "Recall": avg_r,
                    "F1-Score": avg_f1,
                    "Time (s)": avg_t,
                    "Type": "Average",
                })

        total_runtime = round(time.time() - start_global, 2)
        print(f"\n" + "=" * 50)
        print(f" RUN COMPLETION: Finished in {total_runtime}s.")
        print(f"=" * 50)

        self.log.append({
            "Track": "ALL_TRACKS",
            "Task": "TOTAL_PROGRAM_TIME",
            "Precision": "",
            "Recall": "",
            "F1-Score": "",
            "Time (s)": total_runtime,
            "Type": "Summary",
        })

        self.results_to_csv(csv_out)

    def results_to_csv(self, filename: str):
        fields = ["Track", "Task", "Precision", "Recall", "F1-Score", "Time (s)", "Type"]
        with open(filename, mode="w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(self.log)
        print(f" Compilation written to: {filename}")


if __name__ == "__main__":
    m = MOSAIC(lex_thres=0.62, sem_thres=0.62)
    runner = OAEITrackRunner(matcher=m)
    runner.run_all_tracks("../tracks", csv_out="mosaic_evaluation_report.csv")


 TRACK RUNNER: ANATOMY

MOSAIC Task: human ➔ mouse
 [MOSAIC] Initializing embedding engine on CPU: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9089.85it/s]
C:\Users\PC\AppData\Local\Temp\ipykernel_17768\2372242641.py:51: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  self.model = quantize_dynamic(raw_model, {torch.nn.Linear}, dtype=torch.qint8)


 Step complete. MOSAIC returned 1799 matches in 19.99s.
   [MOSAIC] Output saved to: results/mosaic_anatomy_human-mouse.ttl
   Metrics -> Precision: 0.6693, Recall: 0.7942, F1-Score: 0.7264 (Time: 19.99s)

 Track [anatomy] AVERAGES -> P: 0.6693, R: 0.7942, F1: 0.7264 | Avg Time: 19.99s

 TRACK RUNNER: BIO-ML

MOSAIC Task: ncit ➔ doid
 Step complete. MOSAIC returned 4951 matches in 27.01s.
   [MOSAIC] Output saved to: results/mosaic_bio-ml_ncit-doid.ttl
   Metrics -> Precision: 0.7154, Recall: 0.7559, F1-Score: 0.7351 (Time: 27.01s)

MOSAIC Task: omim ➔ ordo
 Step complete. MOSAIC returned 3416 matches in 26.92s.
   [MOSAIC] Output saved to: results/mosaic_bio-ml_omim-ordo.ttl
   Metrics -> Precision: 0.5246, Recall: 0.4816, F1-Score: 0.5022 (Time: 26.92s)

MOSAIC Task: snomed.body ➔ fma.body
 Step complete. MOSAIC returned 14332 matches in 201.27s.
   [MOSAIC] Output saved to: results/mosaic_bio-ml_snomed.body-fma.body.ttl
   Metrics -> Precision: 0.1074, Recall: 0.2121, F1-Score: 0.14

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001FEB19B9850>
Traceback (most recent call last):
  File "c:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\rdflib\xsd_datetime.py", line 586, in parse_xsd_date
    raise ValueError("XSD Date string must contain at least two dashes")
ValueError: XSD Date string must contain at least two dashes
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001FEB19B9850>
Traceback (most recent call last):
  File "c:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: 


MOSAIC Task: dha ➔ unesco


Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001FEB19B9850>
Traceback (most recent call last):
  File "c:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\rdflib\xsd_datetime.py", line 586, in parse_xsd_date
    raise ValueError("XSD Date string must contain at least two dashes")
ValueError: XSD Date string must contain at least two dashes
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x000001FEB19B9850>
Traceback (most recent call last):
  File "c:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: 

 Step complete. MOSAIC returned 5 matches in 0.53s.
   [MOSAIC] Output saved to: results/mosaic_digital-humanities_dha-unesco.ttl
   Metrics -> Precision: 0.0, Recall: 0.0, F1-Score: 0.0 (Time: 0.53s)

MOSAIC Task: idai ➔ pactols
 Step complete. MOSAIC returned 20 matches in 0.48s.
   [MOSAIC] Output saved to: results/mosaic_digital-humanities_idai-pactols.ttl
   Metrics -> Precision: 0.0, Recall: 0.0, F1-Score: 0.0 (Time: 0.48s)

MOSAIC Task: idai ➔ parthenos
 Step complete. MOSAIC returned 74 matches in 0.59s.
   [MOSAIC] Output saved to: results/mosaic_digital-humanities_idai-parthenos.ttl
   Metrics -> Precision: 0.0, Recall: 0.0, F1-Score: 0.0 (Time: 0.59s)

MOSAIC Task: ironagedanube ➔ pactols
 Step complete. MOSAIC returned 35 matches in 0.44s.
   [MOSAIC] Output saved to: results/mosaic_digital-humanities_ironagedanube-pactols.ttl
   Metrics -> Precision: 0.0, Recall: 0.0, F1-Score: 0.0 (Time: 0.44s)

MOSAIC Task: oeai ➔ parthenos
 Step complete. MOSAIC returned 63 matches in 0

KeyboardInterrupt: 